# Distribution Shift (Camelyon17-WILDS)

A first **domain-generalization** study. Camelyon17 has 96x96 tumor-vs-normal patches from five hospitals. The WILDS split trains on some hospitals and tests on held-out ones, so the headline number is **out-of-distribution** accuracy, not in-distribution. This exposes the shortcut-learning failure mode directly: a model can ace the training hospitals and fall apart on a new scanner. This is a scaffold; the data access and the split explanation are here, and the model plus WILDS group evaluation are the next steps.

> The dataset is large (~10 GB) and WILDS is PyTorch-native, so run this on Colab.

## Setup

In [ ]:
# One-time setup: make the `visualization` helper importable, then fetch data +
# resolve paths. Each study's fetch logic lives in its own download_data.py.
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.isdir("ConvolutedComputerVision"):
        !git clone -q https://github.com/samlowe106/ConvolutedComputerVision.git
    sys.path.insert(0, "ConvolutedComputerVision/src")

from visualization import colab_bootstrap

DATA_ROOT, CKPT_ROOT = colab_bootstrap(
    study="camelyon17-wilds",
    pip_packages=("pooch", "seaborn", "wilds"),
)

In [ ]:
import datetime

from wilds import get_dataset

notebook_start_time = datetime.datetime.now()

## Data: hospitals as domains

In [ ]:
dataset = get_dataset(dataset="camelyon17", root_dir=DATA_ROOT, download=True)

# WILDS splits: train on some hospitals, id_val on the same, test on held-out ones (OOD)
train_data = dataset.get_subset("train")
id_val_data = dataset.get_subset("id_val")
test_data = dataset.get_subset("test")  # out-of-distribution hospitals
for name, subset in [
    ("train", train_data),
    ("id_val", id_val_data),
    ("test (OOD)", test_data),
]:
    print(f"{name:<12} {len(subset):>7} patches")

## Model and evaluation (next steps)

WILDS is PyTorch-native, so the natural path here is a torchvision CNN (for example a small ResNet) trained on `get_train_loader(train_data)`, then evaluated with `dataset.eval(preds, y, metadata)`, which reports accuracy **per hospital** and the worst-group number. The point of the study is the gap between in-distribution (`id_val`) and out-of-distribution (`test`) accuracy, and whether techniques like stronger augmentation or GroupDRO close it.

In [ ]:
# TODO: build a torchvision model, train on get_train_loader(train_data), and evaluate
# out-of-distribution with dataset.eval(...). Report id_val vs test (OOD) accuracy.

In [ ]:
notebook_end_time = datetime.datetime.now()
print(
    f"Notebook last run (end-to-end): {notebook_end_time} "
    f"(duration: {notebook_end_time - notebook_start_time})"
)